<a href="https://colab.research.google.com/github/vikashvikku/SEO_Implementation/blob/main/SEO_Paper_Imp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install gymnasium stable-baselines3 numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 2.9 MB/s eta 0:00:00


In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

In [3]:
class SEOEnvironment(gym.Env):
    """
    AI-Driven Reinforcement Learning Environment
    for SEO and Voice Search Optimization
    """

    metadata = {"render_modes": []}

    def __init__(self):
        super().__init__()

        # State:
        # [SERP rank, CTR, Bounce Rate, Dwell Time, Voice Query Score]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5,), dtype=np.float32
        )

        # Actions:
        # 0: Optimize title
        # 1: Add FAQ (voice optimization)
        # 2: Improve content depth
        # 3: Add schema markup
        self.action_space = spaces.Discrete(4)

        self.state = None
        self.steps = 0
        self.max_steps = 50

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.random.rand(5).astype(np.float32)
        self.steps = 0
        return self.state, {}

    def step(self, action):
        serp, ctr, bounce, dwell, voice = self.state

        # Action effects
        if action == 0:      # Title optimization
            ctr += 0.05
        elif action == 1:    # FAQ (voice)
            voice += 0.10
            ctr += 0.03
        elif action == 2:    # Content depth
            dwell += 0.10
            bounce -= 0.05
        elif action == 3:    # Schema markup
            ctr += 0.04

        # Clamp values
        ctr = np.clip(ctr, 0, 1)
        bounce = np.clip(bounce, 0, 1)
        dwell = np.clip(dwell, 0, 1)
        voice = np.clip(voice, 0, 1)

        # Reward (Digital Marketing Goal)
        reward = (2 * ctr) + dwell - bounce

        # SERP rank improves as reward increases
        serp = np.clip(serp - reward * 0.05, 0, 1)

        self.state = np.array([serp, ctr, bounce, dwell, voice], dtype=np.float32)

        self.steps += 1
        terminated = self.steps >= self.max_steps
        truncated = False

        return self.state, reward, terminated, truncated, {}


In [4]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
env = make_vec_env(SEOEnvironment, n_envs=1)

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=0.001,
    gamma=0.99,
    buffer_size=50000,
    batch_size=64,
    exploration_fraction=0.1,
    verbose=1
)

model.learn(total_timesteps=20000)


Using cpu device
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 50       |
|    ep_rew_mean      | 123      |
|    exploration_rate | 0.905    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 689      |
|    time_elapsed     | 0        |
|    total_timesteps  | 200      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 1.63     |
|    n_updates        | 24       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 50       |
|    ep_rew_mean      | 123      |
|    exploration_rate | 0.81     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 813      |
|    time_elapsed     | 0        |
|    total_timesteps  | 400      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.106    |
|  

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 50       |
|    ep_rew_mean      | 123      |
|    exploration_rate | 0.715    |
| time/               |          |
|    episodes         | 12       |
|    fps              | 845      |
|    time_elapsed     | 0        |
|    total_timesteps  | 600      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.0129   |
|    n_updates        | 124      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 50       |
|    ep_rew_mean      | 125      |
|    exploration_rate | 0.62     |
| time/               |          |
|    episodes         | 16       |
|    fps              | 825      |
|    time_elapsed     | 0        |
|    total_timesteps  | 800      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.00328  |
|    n_updates      

In [6]:
obs = env.reset()
total_reward = 0

for _ in range(50):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = env.step(action)
    total_reward += reward
    if done:
        break

print("Total Optimization Reward:", total_reward)


Total Optimization Reward: [130.94563]


In [7]:
# This step is totally optional, as it is just showing how learning process is working and logging the rewards per episode
episode_rewards = []

for episode in range(10):
    obs = env.reset()
    total_reward = 0

    for _ in range(50):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        if done:
            break

    episode_rewards.append(total_reward.item())

print("Episode Rewards:", episode_rewards)



Episode Rewards: [143.01708984375, 141.23138427734375, 128.55699157714844, 134.1027374267578, 114.54959869384766, 135.3614044189453, 141.33282470703125, 116.04054260253906, 145.3065643310547, 132.9829559326172]
